# 03 — Segmentation Training

Train **one fold** of one segmentation experiment, end-to-end. Everything about a run — model, loss, optimizer, scheduler, augmentation, fold — is in the `EXPERIMENT` dict in cell 3.

To switch experiments: edit cell 3, run all cells again. To train all 5 folds, change `EXPERIMENT["fold"]` and re-run from cell 3 (no kernel restart needed), or use the multi-fold loop at the bottom.

**Outputs per fold (after the final sync cell):**

```
senior_project/
├── configs/                            ← NEW
│   ├── __init__.py
│   └── seg/
│       ├── __init__.py
│       └── reference_experiments.py    ← the 7 §13 recipes
├── src/                                (unchanged)
└── notebooks/
    └── segmentation/
        └── 03_train.ipynb              ← cell 3 updated
```

**Runtime:** open on **GPU runtime** (T4 or better). One fold of `01_dice_image_level` typically takes 30–60 minutes.

## Cell 1 — Install dependencies

In [ ]:
%pip install -q pytorch-lightning segmentation-models-pytorch albumentations opencv-python-headless timm

## Cell 2 — Bootstrap: Drive + repo + local SSD scratch

In [ ]:
import os, sys

if not os.path.exists("/content/senior_project"):
    from google.colab import userdata
    try:
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None
    url = "https://github.com/salemaker47/senior_project.git"
    if token:
        url = url.replace("https://", f"https://{token}@", 1)
    os.system(f"git clone {url} /content/senior_project")
if "/content/senior_project" not in sys.path:
    sys.path.insert(0, "/content/senior_project")

from src.notebook_setup import setup_environment, copy_to_local

DRIVE_ROOT, REPO_ROOT = setup_environment(
    repo_url="https://github.com/salemaker47/senior_project.git",
)
LOCAL_ROOT   = copy_to_local(DRIVE_ROOT, datasets=["figshare"])
PROJECT_ROOT = LOCAL_ROOT   # training writes go to fast local SSD

print(f"DRIVE_ROOT:   {DRIVE_ROOT}")
print(f"REPO_ROOT:    {REPO_ROOT}")
print(f"LOCAL_ROOT:   {LOCAL_ROOT}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

## Cell 3 — EXPERIMENT dict (the single source of truth)

Edit this dict to define a run. Defaults match the FigShare reference reproduction's experiment 01.

**Common edits when iterating:**
- `fold`: 1 → 2 → 3 → 4 → 5 (run all 5)
- `name`: switch between the 7 reference experiments (see §13 of the project instruction)
- `loss_name`, `model_name`: swap registries
- `max_epochs`: drop to 3–5 for a quick smoke test before a real run

In [ ]:
EXPERIMENT = {
    "name":         "01_dice_image_level",
    "task":         "segmentation",
    "dataset":      "figshare",
    "split_scheme": "image_level",

    "fold": 1,
    "image_size": 256,
    "batch_size": 8,
    "num_workers": 2,

    "preprocessing":         "original",
    "augmentation_strength": "reference",

    "model_name":      "smp_unet_resnet34",
    "encoder_weights": "imagenet",

    "loss_name":   "dice",
    "loss_kwargs": {},

    "optimizer_name":   "adam",
    "optimizer_kwargs": {"lr": 1e-4},
    "scheduler_name":   "reduce_on_plateau",
    "scheduler_kwargs": {"mode": "min", "factor": 0.1, "patience": 5, "min_lr": 1e-7},
    "scheduler_monitor": "val_loss",

    "metric_kind": "micro_macro",
    "max_epochs": 100,
    "patience":   15,
    "threshold":  0.5,
    "seed":       42,
}

# Cell-3 sanity check from project instruction §6
assert EXPERIMENT["task"] == "segmentation"
assert EXPERIMENT["dataset"] in ("figshare", "brats2024")
assert EXPERIMENT["split_scheme"] in ("image_level", "patient_level")
assert EXPERIMENT["fold"] in (1, 2, 3, 4, 5)

print(f"EXPERIMENT: {EXPERIMENT['name']} (fold {EXPERIMENT['fold']}) on {EXPERIMENT['dataset']}/{EXPERIMENT['split_scheme']}")

## Cell 4 — Seed + path resolution

In [ ]:
from src.train_utils import set_global_seed
from src.file_utils import experiment_paths, fold_split_csv_paths

set_global_seed(EXPERIMENT["seed"])

paths = experiment_paths(
    LOCAL_ROOT,
    task=EXPERIMENT["task"],
    dataset=EXPERIMENT["dataset"],
    experiment_name=EXPERIMENT["name"],
    fold=EXPERIMENT["fold"],
)
csv_paths = fold_split_csv_paths(
    LOCAL_ROOT,
    dataset=EXPERIMENT["dataset"],
    scheme=EXPERIMENT["split_scheme"],
    fold=EXPERIMENT["fold"],
)

for k, v in paths.items():
    print(f"  {k}: {v}")
print()
for k, v in csv_paths.items():
    print(f"  csv_{k}: {v}")

## Cell 5 — Load fold CSVs

In [ ]:
from src.data_utils import load_metadata, validate_metadata

train_df = load_metadata(csv_paths["train"])
val_df   = load_metadata(csv_paths["val"])

validate_metadata(train_df)
validate_metadata(val_df)

print(f"train: {len(train_df):>5} images, {train_df['patient_id'].nunique()} patients")
print(f"val:   {len(val_df):>5} images, {val_df['patient_id'].nunique()} patients")
print(f"\nclass balance (train):")
print(train_df["tumor_class"].value_counts().to_string())

## Cell 6 — Dataloaders

In [ ]:
from src.sg_data_utils import build_dataloaders

train_loader, val_loader = build_dataloaders(
    train_df=train_df,
    val_df=val_df,
    project_root=LOCAL_ROOT,
    batch_size=EXPERIMENT["batch_size"],
    num_workers=EXPERIMENT["num_workers"],
    image_size=EXPERIMENT["image_size"],
    preprocessing=EXPERIMENT["preprocessing"],
    augmentation_strength=EXPERIMENT["augmentation_strength"],
    seed=EXPERIMENT["seed"],
)

# Pull one batch to confirm shapes
imgs, masks = next(iter(train_loader))
print(f"train batch: image={tuple(imgs.shape)}, mask={tuple(masks.shape)}, mask unique={masks.unique().tolist()}")

## Cell 7 — Build model, loss, Lightning module

In [ ]:
from src.sg_models           import build_model, count_parameters
from src.sg_losses           import get_loss
from src.sg_lightning_module import BrainTumorSegModule

model = build_model(
    name=EXPERIMENT["model_name"],
    encoder_weights=EXPERIMENT["encoder_weights"],
)
params_count = count_parameters(model)
print(f"model: {EXPERIMENT['model_name']}  ({params_count:,} trainable params)")

loss_fn = get_loss(EXPERIMENT["loss_name"], **EXPERIMENT["loss_kwargs"])
print(f"loss:  {EXPERIMENT['loss_name']}  -> {type(loss_fn).__name__}")

module = BrainTumorSegModule(
    model=model,
    loss_fn=loss_fn,
    threshold=EXPERIMENT["threshold"],
    optimizer_name=EXPERIMENT["optimizer_name"],
    optimizer_kwargs=EXPERIMENT["optimizer_kwargs"],
    scheduler_name=EXPERIMENT["scheduler_name"],
    scheduler_kwargs=EXPERIMENT["scheduler_kwargs"],
    scheduler_monitor=EXPERIMENT["scheduler_monitor"],
    metric_kind=EXPERIMENT["metric_kind"],
)

## Cell 8 — Callbacks + Trainer

In [ ]:
from src.train_utils import build_callbacks, build_trainer, TrainingTimingCallback

callbacks = build_callbacks(
    ckpt_dir=paths["checkpoints"],
    monitor="val_dice",
    mode="max",
    patience=EXPERIMENT["patience"],
)
trainer = build_trainer(
    callbacks=callbacks,
    log_dir=paths["logs"],
    max_epochs=EXPERIMENT["max_epochs"],
    precision="auto",
)
print(f"trainer: precision={trainer.precision}, max_epochs={trainer.max_epochs}, "
      f"callbacks={[type(c).__name__ for c in callbacks]}")

## Cell 9 — Train

This is the slow cell. With `max_epochs=100` and `patience=15` on a T4 GPU, expect 30–60 minutes for the reference experiment. Lightning will print a progress bar for each epoch and call `EarlyStopping` when `val_dice` stalls.

In [ ]:
trainer.fit(module, train_dataloaders=train_loader, val_dataloaders=val_loader)

## Cell 10 — Collect training meta (Enhancement E)

In [ ]:
import torch
from pytorch_lightning.callbacks import ModelCheckpoint

ckpt_cb   = next(c for c in callbacks if isinstance(c, ModelCheckpoint))
timing_cb = next(c for c in callbacks if isinstance(c, TrainingTimingCallback))

# Read best epoch + best val_dice from the checkpoint blob
best_ckpt_path = ckpt_cb.best_model_path
best_blob = torch.load(best_ckpt_path, map_location="cpu", weights_only=False)

training_meta = {
    "fold":                 EXPERIMENT["fold"],
    "best_epoch":           int(best_blob.get("epoch", -1)),
    "best_val_dice":        float(ckpt_cb.best_model_score) if ckpt_cb.best_model_score is not None else None,
    "total_epochs_trained": int(trainer.current_epoch + 1),
    "train_seconds":        timing_cb.train_seconds,
    "params_count":         int(params_count),
    "peak_gpu_mem_mb":      timing_cb.peak_gpu_mem_mb,
    "best_ckpt_path":       best_ckpt_path,
}
print("training_meta:")
for k, v in training_meta.items():
    print(f"  {k}: {v}")

## Cell 11 — Export plain `best_model.pt` (no Lightning needed to reload)

In [ ]:
from src.train_utils import export_plain_state_dict

pt_path = export_plain_state_dict(
    lightning_ckpt_path=best_ckpt_path,
    out_pt_path=paths["best_model"],
    extra_meta={
        "experiment":      EXPERIMENT,
        "model_name":      EXPERIMENT["model_name"],
        "encoder_weights": EXPERIMENT["encoder_weights"],
    },
)
print(f"wrote {pt_path}")

## Cell 12 — Save `experiment_config.json` (Enhancement F repro stamp + Enhancement E training meta)

In [ ]:
from src.train_utils import gather_repro_metadata
from src.file_utils  import save_json

repro = gather_repro_metadata(repo_root=REPO_ROOT)

experiment_config = {
    "EXPERIMENT":     EXPERIMENT,
    "training_meta":  training_meta,
    "repro_metadata": repro,
}
save_json(experiment_config, paths["experiment_config_json"])
print(f"wrote {paths['experiment_config_json']}")

## Cell 13 — Sync this fold's outputs to Drive

In [ ]:
from src.notebook_setup import sync_outputs_to_drive

sync_outputs_to_drive(
    drive_root=DRIVE_ROOT,
    local_root=LOCAL_ROOT,
    task=EXPERIMENT["task"],
    dataset=EXPERIMENT["dataset"],
    experiment_name=EXPERIMENT["name"],
    categories=["checkpoints", "logs"],
)
print("done")

## Done — what to do next

The fold's checkpoint, plain state dict, config, and Lightning logs are now in Drive at:

```
Drive/Senior_Project/outputs/{checkpoints,logs}/segmentation/<dataset>/<exp>/fold_<k>/
```

### To train another fold of the same experiment

1. Scroll back to **Cell 3**, change `"fold": 1` to `"fold": 2`.
2. Optionally **Runtime → Restart runtime** to clear GPU memory (otherwise the previous fold's model occupies VRAM).
3. **Runtime → Run all** (or run from Cell 4 onward — Cell 2's bootstrap is idempotent).

### To train all 5 folds in one session (advanced)

Wrap cells 4–13 in a function and loop, or use this minimal pattern in a new cell:

```python
for fold in (1, 2, 3, 4, 5):
    EXPERIMENT["fold"] = fold
    # re-run cells 4-13 (each cell's logic is self-contained on EXPERIMENT)
```

### To switch experiments (e.g. exp 02 -> exp 03)

1. Change `"name"` and `"loss_name"` (and any other relevant key) in Cell 3.
2. Run all cells again from Cell 3. Outputs land in a separate `<exp>` directory, so old runs aren't clobbered.

Next: **NB04 (`04_results.ipynb`)** for qualitative figures and **NB05 (`05_test.ipynb`)** for the quantitative eval + Enhancements A–F tables.